In [66]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [67]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    strike_rate: float
    bpb: float
    boundary_percentage: float
    summary: str

In [ ]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs'] / state['balls']) * 100
    return {"strike_rate": sr}

In [69]:
def calculate_bpb(state: BatsmanState):
    bpb = state['balls'] / (state['fours'] + state["sixes"])
    return {'bpb': bpb}

In [70]:
def calculate_bp(state: BatsmanState):
    bp = (((state['fours'] * 4) + (state["sixes"] * 6)) / state["runs"]) * 100
    return {"boundary_percentage": bp}

In [ ]:
def summary(state: BatsmanState):
    summary = f"""
    Strike rate: {state['strike_rate']} \n
    Balls per boundary: {state['bpb']} \n
    Boundary percentage: {state['boundary_percentage']} \n
    """
    state["summary"] = summary
    return state    

In [72]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_bp', calculate_bp)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_bp')

graph.add_edge('calculate_sr', "summary")
graph.add_edge('calculate_bpb', "summary")
graph.add_edge('calculate_bp', "summary")

graph.add_edge("summary", END)

workflow = graph.compile()


In [73]:
initial_state = {
    "runs": 100,
    "balls": 50,
    "fours": 6,
    "sixes": 4
}

output = workflow.invoke(initial_state)

print(output)

{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'strike_rate': 0.02, 'bpb': 5.0, 'boundary_percentage': 48.0, 'summary': '\n    Strike rate: 0.02 \n\n    Balls per boundary: 5.0 \n\n    Boundary percentage: 48.0 \n\n    '}
